# 🍐 Little Fig — CPU/GPU Native LLM Training\n\n**Train language models on any hardware — even 8GB RAM.**\n\n| Feature | Result |\n|---|---|\n| Quantization quality | Beats NF4 on 156/156 TinyLlama layers (+5.4% MSE) |\n| GPU training speed | **7× faster** than BnB NF4 QLoRA |\n| FigMeZO optimizer | −18.6% loss vs standard MeZO |\n| Sensitivity LISA | −10% loss vs random layer selection |\n| Memory Fabric | Weight-space memory with gating + decay |\n\n**License:** AGPL-3.0 (open source, commercial license available)\n**Author:** 0xticketguy / Harboria Labs

In [ ]:
# Install\n!pip install torch --quiet\n!pip install git+https://github.com/ticketguy/littlefig.git#egg=little-fig[train] --quiet\nprint('✅ Little Fig installed')

In [ ]:
# Check GPU\nimport torch\nprint(f'PyTorch {torch.__version__}')\nprint(f'CUDA: {torch.cuda.is_available()}')\nif torch.cuda.is_available():\n    print(f'GPU: {torch.cuda.get_device_name()}')\n    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Quick Start: Fine-tune TinyLlama with FigQuant

In [ ]:
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig\nfrom little_fig.engine.tier import TrainingTier\n\n# Load model with FigQuant INT4 quantization + LoRA\nmodel = FigModel.from_pretrained(\n    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',\n    lora_r=16,\n    lora_alpha=32,\n    shared_codebook=True,  # 5× faster loading\n)\nprint(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params')

In [ ]:
# Train on Alpaca\nconfig = FigTrainingConfig(\n    num_epochs=1,\n    learning_rate=2e-4,\n    max_seq_length=512,\n    batch_size=4,\n    gradient_accumulation_steps=4,\n    logging_steps=10,\n)\n\ntrainer = FigTrainer(model, config)\ntrainer.load_dataset('tatsu-lab/alpaca', max_samples=500)\ntrainer.train()

In [ ]:
# Save adapter (tiny — ~5MB)\nmodel.save_adapter('./my_adapter')\nprint('✅ Adapter saved!')

## Memory Fabric (Weight-Space Memory)

In [ ]:
# Load with Memory Fabric — the model REMEMBERS\nmodel = FigModel.from_pretrained(\n    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',\n    lora_r=16,\n    memory_fabric=True,  # Enable dual-architecture memory\n    shared_codebook=True,\n)\n\n# Write memories into the weights\nmodel.write_memory('personal', 'The user prefers Python for backend work.')\nmodel.write_memory('wiki', 'The speed of light is 299,792,458 m/s.')\n\n# Check what the model holds\nprint(model.memory_confidence())

## FigMeZO (Error-Shaped Zeroth-Order Optimizer)\n\nOriginal research: −18.6% loss improvement vs standard MeZO.\nProbes clean dimensions harder, noisy dimensions lighter.

In [ ]:
from little_fig.engine.figmezo import FigMeZO, FigMeZOConfig\n\n# Use FigMeZO when you can't afford backward passes\noptimizer = FigMeZO(model.model, FigMeZOConfig(\n    learning_rate=1e-5,\n    epsilon=1e-3,\n    shaping_strength=-0.3,  # Negative = inverse shaping (our finding)\n))\n\n# Train with only forward passes — no gradients needed!\nfor step in range(10):\n    loss = optimizer.step(lambda: model(\n        input_ids=torch.randint(0, 32000, (1, 64)).cuda(),\n        labels=torch.randint(0, 32000, (1, 64)).cuda()\n    ).loss)\n    if step % 5 == 0: print(f'Step {step}: loss={loss:.4f}')

## Run CogMemBench\n\n5-axis cognitive memory benchmark. Evaluate any model.

In [ ]:
import sys; sys.path.insert(0, '.')\n!git clone https://github.com/ticketguy/littlefig.git /tmp/lf --quiet 2>/dev/null\nsys.path.insert(0, '/tmp/lf')\n\nfrom cogmembench import CogMemRunner\n\nrunner = CogMemRunner(per_axis=10)  # Small run for demo\nresults = runner.run(\n    model_fn=lambda prompt: 'I am not sure about this.',  # Replace with real model\n    max_cases=50,\n)\nprint(f'CogMem Score: {results["cogmem_score"]}/100')